In [ ]:
# BLOCK 0 – GLOBAL SETUP: imports ▸ paths ▸ flags ▸ constants
# ──────────────────────────────────────────────────────────────

import gc
import traceback
from pathlib import Path

import cv2
import nibabel as nib
import numpy as np
import pandas as pd
import torch
from totalsegmentator.python_api import totalsegmentator
from ultralytics import YOLO
from tqdm.auto import tqdm
import shutil, time

# ── PATHS (EDIT HERE) ────────────────────────────────────────
MODEL_PATH = Path(r"\\Cifshd\homedir$\Ryan\model\best.pt")
NIFTI_DIR = Path(r"\\Cifshd\homedir$\Ryan\testingtestingfull")
CSV_PATH = NIFTI_DIR / "overscanning_results.csv"

# ── FLAGS ─────────────────────────────────────────────────────
DISPLAY_DETECTION = True
FAST_MODEL = False
MULTI_LABEL_MASK = True

# ── CONSTANTS ─────────────────────────────────────────────────
FINAL_CONF = 0.20
BACKGROUND_HU = -300
model = YOLO(str(MODEL_PATH))

In [ ]:
#  Pubic-symphysis detection ➔ caudal-overscan  (robust, femur-aware)
# ────────────────────────────────────────────────────────────────────────────────
# 0)  SCANS ALREADY IN CSV
# ────────────────────────────────────────────────────────────────────────────────

if CSV_PATH.exists():
    done_df  = pd.read_csv(CSV_PATH)
    done_set = set(done_df["file_name"].tolist())
    print(f"↪️  {len(done_set)} rows already in CSV – they’ll be skipped\n")
else:
    done_set = set()

# ────────────────────────────────────────────────────────────────────────────────
# 1)  HELPER FUNCTIONS
# ────────────────────────────────────────────────────────────────────────────────
def preprocess_slice(arr: np.ndarray) -> np.ndarray:
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    arr = (arr * 255).astype(np.uint8)
    return cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR)


def ensure_femur_mask(ct_path: Path) -> Path | None:
    """
    Creates / returns a merged femur mask (label 1). Returns None on TS failure.
    """
    out_dir     = ct_path.parent / "ts_femur"
    fem_l_path  = out_dir / "femur_left.nii.gz"
    fem_r_path  = out_dir / "femur_right.nii.gz"
    merged_path = ct_path.parent / "femur_combined.nii.gz"

    if merged_path.exists():
        return merged_path

    # ── TotalSegmentator with GPU → CPU fallback ────────────────────────────
    if not (fem_l_path.exists() and fem_r_path.exists()):
        out_dir.mkdir(exist_ok=True)
        for dev in ("gpu", "cpu"):
            try:
                totalsegmentator(
                    ct_path, out_dir,
                    roi_subset=["femur_left", "femur_right"],
                    task="total",
                    fast=FAST_MODEL,
                    device=dev,
                )
                break                              # success
            except Exception as e:
                print(f"⚠️  TS({dev}) {ct_path.name}: {e}")
        else:
            return None                            # both runs failed

    # ── merge masks ─────────────────────────────────────────────────────────
    try:
        fem_l = nib.load(fem_l_path).get_fdata() > 0
        fem_r = nib.load(fem_r_path).get_fdata() > 0
    except FileNotFoundError:
        return None

    merged = (fem_l | fem_r).astype(np.uint8)
    if not merged.any():
        return None                                # no femurs in slice range

    ref = nib.load(fem_l_path if fem_l_path.exists() else fem_r_path)
    nib.save(nib.Nifti1Image(merged, ref.affine, ref.header), merged_path)

    for p in (fem_l_path, fem_r_path):  # clean up
        if p.exists():
            p.unlink()

    if out_dir.exists() and not any(out_dir.iterdir()):
        out_dir.rmdir()
    return merged_path


def femur_top_info(ct_path: Path) -> tuple[int, float] | None:
    """
    Returns (slice_idx, world-z_mm) of the cranial-most femur voxel,
    or None if no femurs are present / segmentation failed.
    """
    m = ensure_femur_mask(ct_path)
    if m is None:
        return None
    mask = nib.load(str(m))
    mask_np = mask.get_fdata() > 0
    slices = np.where(mask_np.any(axis=(0, 1)))[0]
    if slices.size == 0:
        return None
    affine = mask.affine
    z_coords = [(k, float((affine @ [0, 0, k, 1])[2])) for k in slices]
    return max(z_coords, key=lambda t: t[1])       # cranial-most


def find_valid_pubic_slice(ct_path: Path, z_cutoff_mm: float) -> int | None:
    """
    Returns the highest-confidence YOLO slice ≤ z_cutoff_mm, or None.
    """
    ct = nib.load(str(ct_path))
    affine = ct.affine
    vol = ct.get_fdata()
    H, W, Z = vol.shape

    best_conf, best_slice = -1.0, None
    for z in range(Z):
        if float((affine @ [0, 0, z, 1])[2]) > z_cutoff_mm:
            continue
        img = preprocess_slice(vol[:, :, z])
        res = model.predict(img, conf=FINAL_CONF, device="cpu", save=False)[0]
        for b in sorted(res.boxes, key=lambda bb: float(bb.conf), reverse=True):
            x1, y1, x2, y2 = b.xyxy[0].tolist()
            if vol[int((y1+y2)/2), int((x1+x2)/2), z] <= BACKGROUND_HU:
                continue
            cx = int((x1+x2)/2)
            if abs(cx - W//2) > 0.20 * W:
                continue
            win = vol[max(0,int((y1+y2)/2)-10):min(H,int((y1+y2)/2)+10),
                      max(0,cx-10):min(W,cx+10), z]
            if win.mean() < 150:
                continue
            conf = float(b.conf)
            if conf > best_conf:
                best_conf, best_slice = conf, z
            break
    return best_slice


# ────────────────────────────────────────────────────────────────────────────────
# 2)  FILTER: keep only original CT volumes
# ────────────────────────────────────────────────────────────────────────────────
def is_ct_vol(p: Path) -> bool:
    if p.parent.name.startswith("ts_"):
        return False
    if p.name.endswith("_combined.nii.gz"):
        return False
    if p.name.startswith(("femur_", "liver_", "spleen_")):
        return False
    return True

nii_paths = [p for p in NIFTI_DIR.rglob("*.nii*") if is_ct_vol(p)]
print(f"🔎 {len(nii_paths)} CT volumes found\n")

# ────────────────────────────────────────────────────────────────────────────────
# 3)  MAIN LOOP
# ────────────────────────────────────────────────────────────────────────────────
for ct_path in nii_paths:
    if ct_path.name in done_set:
        continue
    try:
        print(f"▶ {ct_path.relative_to(NIFTI_DIR.parent)}")

        # (a) femur segmentation
        fem_data = femur_top_info(ct_path)
        if fem_data:
            fem_slice, fem_top_z = fem_data
            z_cut = fem_top_z
        else:
            fem_slice, fem_top_z = None, np.nan
            z_cut = float("inf")                   # no gating

        # (b) YOLO detection with gating
        pubic_slice = find_valid_pubic_slice(ct_path, z_cut)
        if pubic_slice is None and fem_slice is not None:
            pubic_slice   = fem_slice             # fallback
            source_label  = "FemurFallback"
        elif pubic_slice is None:                 # no femur + no YOLO
            print("   ⚠️  no pubic symphysis found – skipping")
            continue
        else:
            source_label  = "YOLO" if not np.isnan(fem_top_z) else "YOLO_NoFemur"

        # (c) caudal overscan metrics
        ct_img  = nib.load(str(ct_path))
        affine  = ct_img.affine
        Z       = ct_img.shape[2]
        pubic_z = float((affine @ [0, 0, pubic_slice, 1])[2])
        end_z   = min(float((affine @ [0, 0, k, 1])[2]) for k in range(Z))
        caudal  = abs(end_z - pubic_z)

        # (d) assemble row (locked column order)
        row = {
            "file_name"         : ct_path.name,
            "pubic_z_mm"        : int(round(pubic_z)),
            "scan_end_z_mm"     : int(round(end_z)),
            "caudal_overscan_mm": int(round(caudal)),
            "femur_top_z_mm"    : (int(round(fem_top_z))
                                   if not np.isnan(fem_top_z) else np.nan),
            "pubic_source"      : source_label,
        }

        # (e) update CSV, preserving cranial columns
        if CSV_PATH.exists():
            df = pd.read_csv(CSV_PATH)
            if row["file_name"] in df["file_name"].values:
                ix = df.index[df["file_name"] == row["file_name"]][0]
                for col in row:
                    df.at[ix, col] = row[col]
            else:
                df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
            df.sort_values("file_name").to_csv(CSV_PATH, index=False)
        else:
            pd.DataFrame([row]).to_csv(CSV_PATH, index=False)

        done_set.add(ct_path.name)
        print("   ✓ saved")

    except Exception as e:
        print("   ⚠️  unexpected error – continuing")
        traceback.print_exc(limit=1)

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\n✅ Finished. CSV now contains {len(done_set)} rows")

In [ ]:
#  BLOCK 3 – Batch liver + spleen segmentation  ▸  NIFTI_DIR
#    • skips any file with “femur_combined” in its name
#    • only processes CTs whose filename begins with the patient-folder name
# ────────────────────────────────────────────────────────────────────────────────

# ── Settings that piggy-back on your Block 0 globals ────────────────────────────
ROOT_DIR = NIFTI_DIR                   # reuse the path you set earlier
DEVICE   = "gpu"                       # change to "cpu" if you ever need to

# ── Helper (unchanged core logic, but uses your FAST_MODEL / MULTI_LABEL_MASK) ─
def ensure_liver_spleen_mask(ct_path: Path) -> Path:
    """
    Guarantees a merged liver-spleen mask in ct_path.parent.
    Returns the mask path; lets any double-failure bubble out
    exactly as before (so downstream try/except still works).
    """
    out_dir      = ct_path.parent / "ts_liver_spleen"
    liver_mask   = out_dir / "liver.nii.gz"
    spleen_mask  = out_dir / "spleen.nii.gz"
    merged_mask  = ct_path.parent / "liver_spleen_combined.nii.gz"

    if merged_mask.exists():
        return merged_mask

    # ── TotalSegmentator with GPU → CPU fallback ──────────────────────────────
    if not (liver_mask.exists() and spleen_mask.exists()):
        out_dir.mkdir(exist_ok=True)
        for dev in ("gpu", "cpu"):
            try:
                totalsegmentator(
                    ct_path,
                    out_dir,
                    roi_subset=["liver", "spleen"],
                    task="total",
                    fast=FAST_MODEL,
                    device=dev,
                )
                break                                       # success
            except Exception as e:
                print(f"⚠️  TS({dev}) {ct_path.name}: {e}")
        else:                                               # both runs failed
            raise RuntimeError("TotalSegmentator failed on GPU and CPU")

    # ── merge & relabel ───────────────────────────────────────────────────────
    liver_data  = nib.load(liver_mask ).get_fdata() > 0
    spleen_data = nib.load(spleen_mask).get_fdata() > 0

    if MULTI_LABEL_MASK:
        combined = np.zeros(liver_data.shape, np.uint8)
        combined[liver_data]  = 1
        combined[spleen_data] = 2
    else:
        combined = (liver_data | spleen_data).astype(np.uint8)

    ref_img = nib.load(liver_mask)
    merged  = nib.Nifti1Image(combined, ref_img.affine, ref_img.header)
    merged.header.set_slope_inter(1, 0)        # zero-slope bugfix
    nib.save(merged, merged_mask)

    shutil.rmtree(out_dir, ignore_errors=True)
    return merged_mask

# ── Walk patient folders and segment where needed ──────────────────────────────
start = time.time()
patients = sorted([p for p in ROOT_DIR.iterdir() if p.is_dir()])
print(f"📂 Found {len(patients)} patient folders under {ROOT_DIR}\n")

for pat in patients:
    try:
        merged_path = pat / "liver_spleen_combined.nii.gz"
        if merged_path.exists():
            print(f"✓ {pat.name} — already has mask, skipping")
            continue

        # candidate CTs: filename starts with folder name, not a femur mask, not temp
        ct_candidates = [
            f for f in pat.glob("*.nii*")
            if f.name.startswith(pat.name)                # matching prefix
            and "femur_combined" not in f.name.lower()    # skip femur masks
            and "liver_spleen_combined" not in f.name.lower()
            and not f.name.startswith("ts_")
        ]

        if not ct_candidates:
            print(f"⚠ {pat.name} — no valid CT volume found, skipping")
            continue

        ct = ct_candidates[0]
        print(f"▶ {pat.name} — segmenting {ct.name} …", end=" ", flush=True)
        ensure_liver_spleen_mask(ct)
        print("done")

    except Exception as e:
        print(f"\n❌ {pat.name} — ERROR: {e}")
        traceback.print_exc()

print(f"\n🏁 All finished in {(time.time()-start)/60:.1f} min")

In [ ]:
# BLOCK 4 – Cranial overscan  (robust, incremental CSV update)
# ────────────────────────────────────────────────────────────────────────────────
# 1)  FILTER: only original CT volumes
# ────────────────────────────────────────────────────────────────────────────────

def is_ct_vol(p: Path) -> bool:
    # skip segmentation sub-folders
    if p.parent.name.startswith("ts_"):
        return False
    # skip any mask file whose OWN filename starts with ts_
    if p.name.startswith("ts_"):
        return False
    # skip merged masks
    if p.name.endswith("_combined.nii.gz"):
        return False
    # skip obvious single-organ masks
    if p.name.startswith(("femur_", "liver_", "spleen_")):
        return False
    return True

ct_paths = [p for p in NIFTI_DIR.rglob("*.nii*") if is_ct_vol(p)]
print(f"🔎 {len(ct_paths)} CT volumes for cranial-overscan pass\n")

# ────────────────────────────────────────────────────────────────────────────────
# 2)  LOAD or CREATE CSV
# ────────────────────────────────────────────────────────────────────────────────
if CSV_PATH.exists():
    csv_df = pd.read_csv(CSV_PATH)
else:
    csv_df = pd.DataFrame(columns=[
        "file_name", "pubic_z_mm", "scan_end_z_mm", "caudal_overscan_mm",
        "femur_top_z_mm", "pubic_source",
        "liver_spleen_z_mm", "scan_start_z_mm", "cranial_overscan_mm", "top_organ"
    ])

# ────────────────────────────────────────────────────────────────────────────────
# 3)  CRANIAL-OVERSCAN FUNCTION
# ────────────────────────────────────────────────────────────────────────────────
def cranial_overscan(ct_path: Path, mask_path: Path) -> tuple[int, int, int, str]:
    ct_img   = nib.load(str(ct_path))
    mask_img = nib.load(str(mask_path))
    affine   = ct_img.affine
    mask_np  = mask_img.get_fdata()

    seg_slices = np.where(mask_np.any(axis=(0, 1)))[0]
    if seg_slices.size == 0:
        raise RuntimeError("combined mask empty")

    z_coords = [(k, float((affine @ [0, 0, k, 1])[2])) for k in seg_slices]

    Z = ct_img.shape[2]
    z_edge0 = float((affine @ [0, 0,      0, 1])[2])
    z_edgeN = float((affine @ [0, 0, Z - 1, 1])[2])
    cranial_edge_z = max(z_edge0, z_edgeN)

    highest_slice, highest_z = min(z_coords, key=lambda t: abs(t[1] - cranial_edge_z))

    labels     = mask_np[:, :, highest_slice][mask_np[:, :, highest_slice] > 0].astype(int)
    organ_map  = {1: "Liver", 2: "Spleen"}
    organ_top  = organ_map.get(int(np.bincount(labels).argmax()), "Unknown")

    cranial_mm    = int(round(abs(cranial_edge_z - highest_z)))
    scan_start_mm = int(round(cranial_edge_z))
    organ_z_mm    = int(round(highest_z))
    return cranial_mm, organ_z_mm, scan_start_mm, organ_top

# ────────────────────────────────────────────────────────────────────────────────
# 4)  MAIN LOOP – incremental CSV write
# ────────────────────────────────────────────────────────────────────────────────
for ct_path in ct_paths:
    mask_path = ct_path.parent / "liver_spleen_combined.nii.gz"
    if not mask_path.exists():
        print(f"⚠️  no liver+spleen mask for {ct_path.name} – skipping")
        continue

    try:
        cranial_mm, organ_z_mm, scan_start_mm, organ_top = cranial_overscan(ct_path, mask_path)
    except Exception as e:
        print(f"⚠️  {ct_path.name}: {e} – skipping")
        continue

    row = {
        "file_name"          : ct_path.name,
        "liver_spleen_z_mm"  : organ_z_mm,
        "scan_start_z_mm"    : scan_start_mm,
        "cranial_overscan_mm": cranial_mm,
        "top_organ"          : organ_top,
    }

    if ct_path.name in csv_df["file_name"].values:
        ix = csv_df.index[csv_df["file_name"] == ct_path.name][0]
        for k, v in row.items():
            csv_df.at[ix, k] = v
    else:
        csv_df = pd.concat([csv_df, pd.DataFrame([row])], ignore_index=True)

    csv_df.sort_values("file_name").to_csv(CSV_PATH, index=False)
    print(f"   ✓ cranial metrics saved for {ct_path.name}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n✅ Cranial pass finished → {CSV_PATH.resolve()}")

In [ ]:
# Preview MP4s of middle 50 coronal slices, aspect-corrected, with tqdm
# ════════════════════════════════════════════════════════════════════════

OUT_DIR = NIFTI_DIR.parent / "trauma_overscan_videos_test"
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
if not {"file_name", "pubic_source"}.issubset(df.columns):
    raise KeyError("CSV missing required columns")

def build_mp4(scan_id: str,
              pubic_z_mm: float,
              organ_z_mm: float,
              organ_label: str,
              pubic_source: str,
              fps: int = 48,
              slice_span: int = 100):
    """Build {scan_id}.mp4 preview"""
    folder   = NIFTI_DIR / scan_id
    ct_path  = next((p for p in folder.glob("*.nii*") if p.stem.startswith(scan_id)), None)
    fem_path = folder / "femur_combined.nii.gz"
    org_path = folder / "liver_spleen_combined.nii.gz"
    mp4_path = OUT_DIR / f"{scan_id}.mp4"
    if not ct_path or not ct_path.exists(): raise FileNotFoundError("CT not found")
    for p in (fem_path, org_path):
        if not p.exists(): raise FileNotFoundError(f"Missing {p.name}")

    ct_img  = nib.load(str(ct_path));  vol = ct_img.get_fdata()
    fem_msk = nib.load(str(fem_path)).get_fdata() > 0
    org_msk = nib.load(str(org_path)).get_fdata() > 0
    affine  = ct_img.affine
    vx, _, vz = ct_img.header.get_zooms()[:3]

    _, Y, Z = vol.shape
    z_world = np.flip((affine @ np.vstack([np.zeros(Z), np.zeros(Z), np.arange(Z), np.ones(Z)]))[2])
    pubic_row = int(np.argmin(np.abs(z_world - pubic_z_mm)))
    organ_row = int(np.argmin(np.abs(z_world - organ_z_mm)))

    mid_y, half = Y // 2, slice_span // 2
    start_y, end_y = max(0, mid_y - half), min(Y - 1, mid_y + half)
    y_stretch = vz / vx

    font, fs, th = cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2
    red, green = (0,0,255), (0,255,0)
    landmark = "Femur" if pubic_source == "FemurFallback" else "Pubic Symphysis"

    def render(y_idx: int):
        ct = np.flipud(vol[:, y_idx, :].T)
        fm = np.flipud(fem_msk[:, y_idx, :].T)
        om = np.flipud(org_msk[:, y_idx, :].T)

        img = np.clip((ct + 200) / 500, 0, 1) * 255
        img = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_GRAY2BGR)

        overlay = np.zeros_like(img)
        overlay[fm] = (255,0,0)     # femur blue
        overlay[om] = (0,255,255)   # organ yellow
        img = cv2.addWeighted(img, 0.8, overlay, 0.25, 0)

        if y_stretch != 1.0:
            h, w = img.shape[:2]
            img = cv2.resize(img, (w, int(h * y_stretch)), interpolation=cv2.INTER_CUBIC)

        h, w = img.shape[:2]
        cv2.line(img, (0, int(pubic_row*y_stretch)), (w-1, int(pubic_row*y_stretch)), red, 2)
        cv2.line(img, (0, int(organ_row*y_stretch)), (w-1, int(organ_row*y_stretch)), green, 2)

        cv2.putText(img, f"{landmark} z={pubic_z_mm:.0f} mm",
                    (10, max(20, int(pubic_row*y_stretch)-6)), font, fs, red, th, cv2.LINE_AA)
        cv2.putText(img, f"{organ_label} z={organ_z_mm:.0f} mm",
                    (10, min(h-10, int(organ_row*y_stretch)+20)), font, fs, green, th, cv2.LINE_AA)
        cv2.putText(img, f"{scan_id} | y={y_idx}",
                    (10, h-10), font, fs, (255,255,0), th, cv2.LINE_AA)
        return img

    first = render(start_y); h, w = first.shape[:2]
    vw = cv2.VideoWriter(str(mp4_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for y in range(start_y, end_y + 1): vw.write(render(y))
    vw.release()

ok = failed = 0
for _, row in tqdm(df.iterrows(), total=len(df), desc="MP4s", unit="scan"):
    sid = row["file_name"].split(".nii")[0]
    try:
        build_mp4(sid,
                  float(row["pubic_z_mm"]),
                  float(row["liver_spleen_z_mm"]),
                  str(row["top_organ"]).strip(),
                  str(row["pubic_source"]).strip())
        ok += 1
    except Exception as e:
        failed += 1
        tqdm.write(f"✗ {sid}: {e}")
        traceback.print_exc()

print(f"\n✓ {ok} videos   ✗ {failed} failed  → {OUT_DIR}")

In [ ]:
# ═════════════════════════════════════════════════════════════
# Overscanning CSV ▸ add yes/no flags, per-row excess columns
#                    total & % overscan (row-level),
#                    plus summary statistics CSV with whole-
#                    number mm averages and % “yes” metrics
# ═════════════════════════════════════════════════════════════
from pathlib import Path
import pandas as pd

# ── Paths ────────────────────────────────────────────────────
CSV_PATH         = Path(r"\\Cifshd\homedir$\Ryan\Trauma_niftis\overscanning_results.csv")
SUMMARY_CSV_PATH = CSV_PATH.with_name("summary_statistics.csv")

# 1 ▸ Load CSV & tidy header ----------------------------------
df = pd.read_csv(CSV_PATH, sep=None, engine="python", encoding="utf-8-sig")
df.columns = df.columns.str.strip().str.replace("\ufeff", "", regex=False)

# 2 ▸ yes/no flag columns -------------------------------------
caudal_thresh = df["pubic_source"].eq("FemurFallback").map({True: 50, False: 30})
df["caudal_overscan?"]  = (df["caudal_overscan_mm"]  > caudal_thresh).map({True: "yes", False: "no"})
df["cranial_overscan?"] = df["cranial_overscan_mm"].gt(30).map({True: "yes", False: "no"})
df["overscanning?"]     = ((df["caudal_overscan?"] == "yes") | (df["cranial_overscan?"] == "yes")).map({True: "yes", False: "no"})

# 3 ▸ Per-row excess distances (…_mm columns) -----------------
df["calc_caudal_overscan_mm"]   = (df["caudal_overscan_mm"]  - caudal_thresh).clip(lower=0)
df["calc_cranial_overscan_mm"]  = (df["cranial_overscan_mm"] - 30).clip(lower=0)

# 4 ▸ Total overscan & whole-number % overscan ----------------
df["calc_total_overscan_mm"] = df["calc_caudal_overscan_mm"] + df["calc_cranial_overscan_mm"]

scan_length  = (df["scan_end_z_mm"] - df["scan_start_z_mm"]).replace(0, pd.NA)
percent_vals = (df["calc_total_overscan_mm"] / scan_length * 100).abs().round()

df["%_overscan"] = percent_vals.apply(
    lambda x: f"{int(x)}%" if pd.notna(x) else pd.NA
)

# 5 ▸ Save updated main CSV (unchanged logic) -----------------
sep = "\t" if "\t" in open(CSV_PATH, encoding="utf-8-sig").readline() else ","
df.to_csv(CSV_PATH, index=False, sep=sep)
print("Main CSV updated ✔️")

# 6 ▸ Summary statistics CSV ----------------------------------
caudal_excess  = df["calc_caudal_overscan_mm"]
cranial_excess = df["calc_cranial_overscan_mm"]
total_excess   = df["calc_total_overscan_mm"]

summary = pd.DataFrame({
    # whole-number averages (ignore rows with 0 mm excess)
    "average_caudal_overscan_excess_mm"  : [int(round(caudal_excess [caudal_excess  > 0].mean(), 0))],
    "average_cranial_overscan_excess_mm" : [int(round(cranial_excess[cranial_excess > 0].mean(), 0))],
    "average_total_overscan_excess_mm"   : [int(round(total_excess  [total_excess   > 0].mean(), 0))],
    # frequency (% “yes”) metrics
    "%_caudal_overscan":  [f"{int(round((df['caudal_overscan?']=='yes').mean() * 100))}%"],
    "%_cranial_overscan": [f"{int(round((df['cranial_overscan?']=='yes').mean() * 100))}%"],
    "%_overscanning":     [f"{int(round((df['overscanning?']=='yes').mean() * 100))}%"],

})

summary.to_csv(SUMMARY_CSV_PATH, index=False)
print(f"Summary statistics written → {SUMMARY_CSV_PATH}")